# Perceptró multi-capa a Scikit

Scikit-learn ofereix dos objectes per treballar amb MLP: MLPClassifier per a problemes de classificació i MLPRegressor per a problemes de regressió. Tots dos comparteixen els mateixos paràmetres i la mateixa interfície, amb l'única diferència que MLPRegressor usa una funció d'activació lineal a la capa de sortida i optimitza una funció de pèrdua de regressió (MSE per defecte), mentre que MLPClassifier adapta automàticament la sortida al tipus de classificació. Aquesta separació segueix la convenció general de scikit-learn de tenir objectes diferenciats per a cada tipus de tasca.


## MLPClassifier

MLPClassifier és la implementació de scikit-learn d'un Multi-Layer Perceptron per a tasques de classificació. Segueix la mateixa interfície que la resta d'estimadors de scikit-learn (`fit`, `predict`, `score`), cosa que facilita integrar-lo en altres programes i comparar-lo amb altres classificadors que hem repassat en el bloc 1. Internament, entrena la xarxa mitjançant l'algorisme de _backpropagation_ i permet configurar-ne l'arquitectura, la funció d'activació i l'algorisme d'optimització.

[Enllaç a la documentació](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)


La manera de construir un objecte d'aquest tipus és la següent:
```python
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation='relu',
    solver='adam',
    batch_size='auto',
    learning_rate='constant',
    learning_rate_init=0.001,
    shuffle=True,
    random_state=None
)
```

### Paràmetres

Anem a veure els paràmetres més rellevants:


- `hidden_layer_sizes` — default: `(100,)`: Defineix el nombre de capes ocultes i el nombre de neurones a cada capa. Cada element de la tupla correspon a una capa oculta.

```python
hidden_layer_sizes=(100,)        # 1 capa oculta amb 100 neurones
hidden_layer_sizes=(100, 50)     # 2 capes ocultes: 100 i 50 neurones
hidden_layer_sizes=(128, 64, 32) # 3 capes ocultes
```

- `activation` — default: `'relu'`

Funció d'activació de les capes ocultes. És important remarcar que la capa de sortida usa Softmax (multiclasse) o Sigmoid (binari) automàticament a partir de com sigui el _groundtruth_.

| Valor | Funció |
|-------|--------|
| `'relu'` | $f(x) = \max(0, x)$ — recomanada per defecte |
| `'tanh'` | $f(x) = \tanh(x)$ |
| `'logistic'` | $f(x) = \sigma(x)$ (Sigmoid) |
| `'identity'` | $f(x) = x$ (sense activació) |

- `solver` — default: `'adam'`

Algorisme d'optimització per actualitzar els pesos.

| Valor | Descripció                                                  |
|-------|-------------------------------------------------------------|
| `'adam'` | Adaptatiu, eficient, recomanat per a la majoria de casos.   |
| `'sgd'` | Gradient descendent estocàstic, requereix més ajust manual. |
| `'lbfgs'` | Basat en quasi-Newton, adequat per a datasets petits.       |

- `batch_size` — default: `'auto'`

Nombre de mostres per actualització de pesos. Amb `'auto'`, scikit-learn usa `min(200, n_samples)`.

```python
batch_size='auto'   # min(200, n_mostres)
batch_size=32       # batch petit → més soroll, pot generalitzar millor
batch_size=256      # batch gran → entrenament més estable i ràpid
```

> **Nota:** `batch_size` només s'aplica quan `solver='sgd'` o `solver='adam'`.

- `learning_rate_init` — default: `0.001`

Valor inicial del ratio d'aprenentatge. Controla la magnitud dels passos d'actualització dels pesos.

```python
learning_rate_init=0.001   # valor per defecte, bon punt de partida
learning_rate_init=0.01    # aprenentatge més ràpid, risc de no convergir
learning_rate_init=0.0001  # aprenentatge lent però estable
```

- `shuffle` — default: `True`

Si aquest paràmetre té el valor `True`, mescla les mostres d'entrenament abans de cada època. Recomanat per evitar que l'ordre de les dades influeixi en l'aprenentatge.

- `random_state` — default: `None`

És la llavor del generador de nombres aleatoris. Fixar-la garanteix la reproductibilitat dels resultats.

```python
random_state=None   # resultats no reproductibles
random_state=42     # resultats reproductibles
```


### Un exemple funcional

A continuació veurem un exemple funcional amb un conjunt de dades de joguina on tenim 1000 observacions amb 10 característiques de 3 classes diferents.

In [ ]:

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

random_state = 42
# Generació del dataset
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_classes=3,
    n_informative=6,
    random_state=random_state
)

# Divisió en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state
)

# Normalització
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Entrenament
model = MLPClassifier(
    hidden_layer_sizes=(64, 32), # Com sabem que són les capes adients?
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=1000,
    random_state=random_state
)
model.fit(X_train, y_train)

# Predicció i avaluació
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


## MLPRegressor

La classe MLPRegressor implementa un perceptró multicapa (MLP) sense cap funció d'activació a la capa de sortida, que és equivalent a usar la funció identitat com a activació. Per tant, utilitza l'error quadràtic com a funció de pèrdua, i la sortida és un conjunt de valors continus. Tal com s'ha explicat anteriorment aquesta classe té els mateixos paràmetres que la classe MLPClassifier.

[Enllaç a la documentació](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html)


La manera de construir un objecte d'aquest tipus és la següent:
```python
from sklearn.neural_network import MLPRegressor

model = MLPRegressor(
    hidden_layer_sizes=(100,),
    activation='relu',
    solver='adam',
    batch_size='auto',
    learning_rate='constant',
    learning_rate_init=0.001,
    shuffle=True,
    random_state=None
)
```

## Exercicis

https://www.kaggle.com/datasets/vinothkannaece/ocean-quality-dataset

1. Donat el conjunt de dades *XXXX* entrena un MLP per predir el valor que tindrà la variable *YYYY*.
2. Troba el millor MLP per resoldre el problema *ZZZZZ*